In [1]:
#### The core script which defines and execute each run based on the following sub-scripts 

# rf_models/
# ├── config.py          # run configs as dicts or dataclasses
# ├── spatial_cv.py      # spatial block splitting logic
# ├── train.py           # fit model given config + data, return results dict
# ├── evaluate.py        # metrics, plots
# ├── importance.py      # feature importance + permutation importance
# └── run.py             # orchestrates everything, saves outputs

from pathlib import Path
import json
import dataclasses
from pathlib import Path

import pandas as pd

from config import RunConfig
from spatial_cv import make_spatial_folds
from train import train_model
from evaluate import evaluate
from importance import get_feature_importance

In [2]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = f'/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Fold_assignments/'

In [3]:
#### DEFINE RUN (MANUAL)

fold = FOLD_DIR + 'capital_folds.csv' ### NEED TO CREATE THIS CSV!!! (see spatial cv script for what's expected)

RUNS = [
    RunConfig(
        run_name         = "capital_rf_v1",
        target           = "log_capital_intensity_USD_per_USD",   # log_capital_intensity_USD_per_USD OR log_labor_intensity_jobs_per_million_USD
        dataset          = "capital.csv",
        fold_assignments = fold,
        model_type       = "rf",
        id_cols          = ["PROJ_ID", "country_ID"],
        spatial_block_col = "country_ID",
    ),
]

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def save_results(results: dict, config: RunConfig, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)

    # config
    (out_dir / "config.json").write_text(
        json.dumps(dataclasses.asdict(config), indent=2)
    )

    # predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # per-fold best params
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # metrics and importance (already DataFrames)
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["importance"].to_csv(out_dir / "importance.csv", index=False)

    print(f"  saved to {out_dir}")


# ── Main ──────────────────────────────────────────────────────────────────────

def run(config: RunConfig):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # resolve feature cols dynamically
    feature_cols = [
        c for c in df.columns
        if c not in config.id_cols + [config.target]
    ]
    print(f"  features: {len(feature_cols)} columns")

    # spatial folds
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # train
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # evaluate
    results["metrics"] = evaluate(results, config)

    # importance
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # save
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)


if __name__ == "__main__":
    for config in RUNS:
        run(config)


════════════════════════════════════════════════════════════
  run: capital_rf_v1
  target: log_capital_intensity_USD_per_USD  |  model: rf
════════════════════════════════════════════════════════════
  features: 34 columns

── Spatial folds ────────────────────────────────────────
  fold_1: 25 train countries (4,929 rows) | 1 test countries (5,182 rows)
  fold_2: 25 train countries (7,403 rows) | 1 test countries (2,708 rows)
  fold_3: 25 train countries (8,827 rows) | 1 test countries (1,284 rows)
  fold_4: 3 train countries (9,174 rows) | 23 test countries (937 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA']
  best params: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_features': 0.5, 'max_depth': 20}

── fold_2 ──────────────────────────────
  test countries: ['USA']
